# Multi-Agent AI Compliance Validator
### AMD Hackathon | vLLM + Qwen3-30B-A3B + Pydantic AI

**Stack**
- **GPU backend** : AMD ROCm + vLLM (Qwen3-30B-A3B served locally)
- **Agent framework** : Pydantic AI (`pydantic-ai`)
- **LLM interface** : OpenAI-compatible REST via `http://localhost:8000/v1`
- **Domains** : BFSI (RBI / SEBI) + Healthcare (HIPAA) + Cross-domain (GDPR)

**Pipeline**
```
Documents / CSVs
      |
  [Agent 1] Document Loader & Chunker
      |
  [Agent 2] Rule Matcher  (keyword scan over regulatory rule bank)
      |
  [Agent 3] Compliance Checker  <-- Qwen3-30B-A3B via vLLM
      |
  [Agent 4] Data Validator  (row-level checks on tabular records)
      |
  [Agent 5] Report Generator  <-- Qwen3-30B-A3B via vLLM
      |
  Audit Report  (JSON + Markdown + CSV)
```


## Step 1 — Launch vLLM Server (AMD GPU)

Run this in a **separate terminal** before executing the notebook.

```bash
# ── Environment setup ──────────────────────────────────────────────────────
# ROCm visible devices (set to your AMD GPU indices, e.g. 0,1)
export HIP_VISIBLE_DEVICES=0,1
export ROCR_VISIBLE_DEVICES=0,1

# Disable Triton flash-attention (required on most AMD cards with vLLM)
export VLLM_USE_TRITON_FLASH_ATTN=0

# ── Serve Qwen3-30B-A3B ────────────────────────────────────────────────────
vllm serve Qwen/Qwen3-30B-A3B \
    --served-model-name Qwen3-30B-A3B \
    --api-key abc-123 \
    --port 8000 \
    --enable-auto-tool-choice \
    --tool-call-parser hermes \
    --trust-remote-code \
    --max-model-len 8192 \
    --gpu-memory-utilization 0.90 \
    --dtype bfloat16
```

> **Tips for AMD ROCm**
> - Use `--dtype bfloat16` — better stability than float16 on ROCm
> - `VLLM_USE_TRITON_FLASH_ATTN=0` avoids Triton kernel issues on AMD
> - `--gpu-memory-utilization 0.90` leaves headroom for activations
> - Qwen3-30B-A3B is a 30B MoE model (only ~3B active params) — fits on a single MI300X

Wait until you see `INFO: Application startup complete` before running cells below.


## Step 2 — Install Dependencies

In [ ]:
# Run once — comment out after first install
# !pip install pydantic-ai openai pandas numpy faker matplotlib
!pip install nest_asyncio -q --root-user-action=ignore

import importlib, subprocess, sys

REQUIRED = {
    "pydantic_ai": "pydantic-ai",
    "openai":      "openai",
    "pandas":      "pandas",
    "numpy":       "numpy",
    "faker":       "faker",
    "matplotlib":  "matplotlib",
    "tabulate":    "tabulate",
}

for module, pkg in REQUIRED.items():
    if importlib.util.find_spec(module) is None:
        print(f"Installing {pkg}...")
        subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=True)
    print(f"  OK  {pkg}")

print("\nAll dependencies ready.")


## Step 3 — Imports

In [ ]:
import os, json, re, hashlib, random, asyncio
from datetime import datetime
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

import pandas as pd
import numpy as np
from faker import Faker

# Pydantic AI
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic import BaseModel

import nest_asyncio
nest_asyncio.apply()   # allows run_sync() inside Jupyter's existing event loop

print("All imports successful.")


## Step 4 — Configure vLLM Connection & Pydantic AI Model

In [ ]:
# ── vLLM endpoint (matches your serve command) ───────────────────────────────
BASE_URL = "http://localhost:8000/v1"
os.environ["BASE_URL"]       = BASE_URL
os.environ["OPENAI_API_KEY"] = "abc-123"

print("Config set:", BASE_URL)

# ── Pydantic AI provider + model ─────────────────────────────────────────────
provider = OpenAIProvider(
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)
agent_model = OpenAIChatModel("Qwen3-30B-A3B", provider=provider)

print(f"Model  : Qwen3-30B-A3B")
print(f"Provider: {provider.__class__.__name__} -> {BASE_URL}")

# ── Health check ─────────────────────────────────────────────────────────────
import urllib.request, json as _json

def check_vllm_health() -> bool:
    try:
        req  = urllib.request.Request(
            "http://localhost:8000/v1/models",
            headers={"Authorization": "Bearer abc-123"}
        )
        with urllib.request.urlopen(req, timeout=5) as resp:
            data   = _json.loads(resp.read())
            models = [m["id"] for m in data.get("data", [])]
            print(f"vLLM running  | models: {models}")
            return True
    except Exception as e:
        print(f"vLLM not reachable ({e}) -- pipeline will run in DRY-RUN mode")
        return False

VLLM_ONLINE = check_vllm_health()


## Step 5 — Regulatory Rule Bank

10 rules across **RBI · SEBI · HIPAA · GDPR** — each with severity, keywords, and remediation steps.


In [ ]:
REGULATORY_RULES = {

    # ── BFSI: Reserve Bank of India ──────────────────────────────────────────
    "RBI_KYC_001": {
        "regulation": "RBI KYC Master Direction 2016",
        "clause":     "Section 16 - Customer Due Diligence",
        "description":"Banks must verify customer identity with valid govt-issued ID before account opening.",
        "keywords":   ["KYC","Aadhaar","PAN","passport","identity verification","CDD"],
        "severity":   "critical",
        "domain":     "BFSI",
        "remediation":"Collect and verify two govt-issued IDs; store copies; log in CDD register.",
    },
    "RBI_AML_002": {
        "regulation": "PMLA 2002 + RBI Guidelines",
        "clause":     "Section 12 - Record Keeping",
        "description":"Financial institutions must maintain transaction records for minimum 5 years.",
        "keywords":   ["AML","transaction record","suspicious transaction","STR","CTR"],
        "severity":   "high",
        "domain":     "BFSI",
        "remediation":"Archive all transaction logs >= 5 years; encrypt at rest; ensure audit trail.",
    },
    "RBI_DATA_003": {
        "regulation": "RBI Data Localisation Framework 2018",
        "clause":     "Para 3 - Payment Data Storage",
        "description":"All payment system data of Indian customers must be stored exclusively in India.",
        "keywords":   ["data localisation","cloud storage","cross-border","payment data","offshore"],
        "severity":   "critical",
        "domain":     "BFSI",
        "remediation":"Migrate data to Indian data-centres; ensure no mirroring abroad.",
    },

    # ── BFSI: SEBI ───────────────────────────────────────────────────────────
    "SEBI_DISC_001": {
        "regulation": "SEBI LODR Regulations 2015",
        "clause":     "Regulation 30 - Material Disclosure",
        "description":"Listed entities must disclose material events to stock exchanges within 24 hours.",
        "keywords":   ["material event","board resolution","disclosure","LODR","BSE","NSE"],
        "severity":   "high",
        "domain":     "BFSI",
        "remediation":"File XBRL disclosure on BSE/NSE within 24h of event.",
    },
    "SEBI_INSIDER_002": {
        "regulation": "SEBI (Prohibition of Insider Trading) Regulations 2015",
        "clause":     "Regulation 9 - Code of Conduct",
        "description":"Companies must maintain an insider trading code of conduct and pre-clearance system.",
        "keywords":   ["insider trading","UPSI","pre-clearance","trading window","designated persons"],
        "severity":   "critical",
        "domain":     "BFSI",
        "remediation":"Implement pre-clearance workflow; close trading windows during results.",
    },

    # ── Healthcare: HIPAA ────────────────────────────────────────────────────
    "HIPAA_PHI_001": {
        "regulation": "HIPAA Privacy Rule 45 CFR Part 164",
        "clause":     "164.502 - Uses and Disclosures of PHI",
        "description":"PHI must not be disclosed without patient authorisation or a permissible purpose.",
        "keywords":   ["PHI","patient data","medical record","health information","disclosure","de-identification"],
        "severity":   "critical",
        "domain":     "Healthcare",
        "remediation":"Obtain signed authorisation form; log all disclosures; apply de-identification.",
    },
    "HIPAA_SEC_002": {
        "regulation": "HIPAA Security Rule 45 CFR Part 164",
        "clause":     "164.312 - Technical Safeguards",
        "description":"ePHI must be encrypted in transit and at rest using NIST-approved algorithms.",
        "keywords":   ["encryption","ePHI","access control","audit log","AES-256","TLS"],
        "severity":   "critical",
        "domain":     "Healthcare",
        "remediation":"Enforce AES-256 at rest; TLS 1.2+ in transit; enable audit logging.",
    },
    "HIPAA_BAA_003": {
        "regulation": "HIPAA Privacy Rule",
        "clause":     "164.504(e) - Business Associate Contracts",
        "description":"Covered entities must have a signed BAA with every vendor that handles PHI.",
        "keywords":   ["BAA","business associate","vendor","third-party","subcontractor"],
        "severity":   "high",
        "domain":     "Healthcare",
        "remediation":"Audit all vendor contracts; execute BAA before any PHI sharing.",
    },

    # ── Cross-domain: GDPR ───────────────────────────────────────────────────
    "GDPR_CONSENT_001": {
        "regulation": "GDPR (EU) 2016/679",
        "clause":     "Article 7 - Conditions for Consent",
        "description":"Personal data may only be processed with freely given, specific, informed consent.",
        "keywords":   ["consent","opt-in","data subject","personal data","DSAR","right to erasure"],
        "severity":   "high",
        "domain":     "Cross-domain",
        "remediation":"Implement granular consent banners; log timestamps; build DSAR workflow.",
    },
    "GDPR_BREACH_002": {
        "regulation": "GDPR Article 33-34",
        "clause":     "Article 33 - Breach Notification",
        "description":"Data breaches must be reported to the supervisory authority within 72 hours.",
        "keywords":   ["data breach","incident","notification","supervisory authority","72 hours"],
        "severity":   "critical",
        "domain":     "Cross-domain",
        "remediation":"Activate incident response plan; notify DPA within 72h.",
    },
}

SEVERITY_SCORE = {"critical": 10, "high": 7, "medium": 4, "low": 1}

df_rules = (pd.DataFrame(REGULATORY_RULES).T
              .reset_index().rename(columns={"index":"rule_id"}))
print(f"Loaded {len(REGULATORY_RULES)} rules across "
      f"{df_rules['domain'].nunique()} domains\n")
print(df_rules[["rule_id","domain","severity"]].to_string(index=False))


## Step 6 — Pydantic Output Schemas

Each agent returns a **typed, validated** object — zero JSON parsing errors.


In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class ComplianceFinding(BaseModel):
    rule_id:      str
    status:       Literal["VIOLATION", "RISK", "COMPLIANT", "NEEDS_REVIEW"]
    confidence:   float = Field(ge=0.0, le=1.0)
    explanation:  str
    remediation:  str
    evidence:     str = ""   # quoted snippet that triggered the finding

class AuditSummary(BaseModel):
    overall_risk:    Literal["CRITICAL", "HIGH", "MEDIUM", "LOW"]
    total_findings:  int
    critical_count:  int
    high_count:      int
    exec_summary:    str
    top_priorities:  list[str]  # top 3 action items

class RemediationPlan(BaseModel):
    rule_id:       str
    priority:      Literal["IMMEDIATE", "SHORT_TERM", "LONG_TERM"]
    owner:         str          # e.g. "DPO", "CISO", "Compliance Team"
    deadline_days: int
    steps:         list[str]

print("Pydantic schemas defined:")
for cls in [ComplianceFinding, AuditSummary, RemediationPlan]:
    fields = list(cls.model_fields.keys())
    print(f"  {cls.__name__:25} fields: {fields}")


## Step 7 — Dataset Generation

Generates synthetic BFSI and Healthcare records that mirror real enterprise data.

> **Drop-in public datasets from Kaggle / PhysioNet:**
>
> | Dataset | Link | Domain |
> |---------|------|--------|
> | IBM AML Transactions (6M rows) | [Kaggle](https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml) | AML/BFSI |
> | Financial Statement Fraud (SEC) | [Kaggle](https://www.kaggle.com/datasets/meirion/financial-statement-fraud-detection-dataset) | BFSI |
> | MIMIC-III Clinical Notes | [PhysioNet](https://physionet.org/content/mimiciii/1.4/) | Healthcare |
> | Synthea Patient Records | [GitHub](https://github.com/synthetichealth/synthea) | Healthcare |
> | GDPR Enforcement Tracker | [enforcementtracker.com](https://www.enforcementtracker.com) | GDPR |
> | OpenFDA Drug Adverse Events | [open.fda.gov](https://open.fda.gov/apis/) | Healthcare |
>
> Replace the CSV paths in `DataValidatorAgent` with the Kaggle/PhysioNet files.


In [ ]:
Path("data/documents").mkdir(parents=True, exist_ok=True)
Path("data/datasets").mkdir(parents=True, exist_ok=True)
Path("reports").mkdir(exist_ok=True)

FAKE = Faker("en_IN")
random.seed(42); np.random.seed(42)

# ── KYC / Customer records (BFSI) ─────────────────────────────────────────────
def make_kyc_records(n: int = 300) -> pd.DataFrame:
    rows = []
    for _ in range(n):
        kyc_ok   = random.random() > 0.15
        rec_ok   = random.random() > 0.10
        india_ok = random.random() > 0.08
        rows.append({
            "customer_id":                   FAKE.uuid4()[:8].upper(),
            "name":                          FAKE.name(),
            "account_type":                  random.choice(["Savings","Current","NRE","Demat"]),
            "kyc_verified":                  kyc_ok,
            "kyc_document":                  random.choice(["Aadhaar","PAN","Passport"]) if kyc_ok else None,
            "transaction_records_maintained":rec_ok,
            "data_stored_in_india":          india_ok,
            "account_opened":                FAKE.date_between("-5y","today").isoformat(),
            "risk_category":                 random.choice(["Low","Medium","High"]),
            "aml_flag":                      random.random() < 0.05,
            "flag": not kyc_ok or not rec_ok or not india_ok,
        })
    return pd.DataFrame(rows)

# ── Patient records (Healthcare) ──────────────────────────────────────────────
def make_patient_records(n: int = 200) -> pd.DataFrame:
    rows = []
    for _ in range(n):
        enc_ok  = random.random() > 0.12
        baa_ok  = random.random() > 0.18
        con_ok  = random.random() > 0.10
        br_ok   = random.random() > 0.05
        rows.append({
            "patient_id":                  f"PT-{FAKE.uuid4()[:6].upper()}",
            "diagnosis_code":              random.choice(["I10","E11","J45","C50","K21"]),
            "phi_encrypted":               enc_ok,
            "baa_signed":                  baa_ok,
            "consent_obtained":            con_ok,
            "breach_reported_within_72h":  br_ok,
            "vendor_count":                random.randint(1, 8),
            "last_audit_date":             FAKE.date_between("-2y","today").isoformat(),
            "flag": not enc_ok or not baa_ok or not con_ok,
        })
    return pd.DataFrame(rows)

# ── Policy documents (free text) ──────────────────────────────────────────────
DOCUMENTS = {
    "data_privacy_policy.txt": (
        "DATA PRIVACY AND SECURITY POLICY - Version 3.2\n"
        "Effective: 1 April 2024\n\n"
        "1. Data Collection\n"
        "   All personal data undergoes KYC verification per RBI KYC Master Direction 2016.\n"
        "   Identity documents (Aadhaar, PAN) are stored in our secure document management system.\n\n"
        "2. Data Storage\n"
        "   Customer data is stored in AWS Mumbai (ap-south-1), complying with RBI data localisation.\n"
        "   No cross-border data transfer occurs without regulatory approval.\n\n"
        "3. Transaction Records\n"
        "   Financial records are archived for minimum 5 years in encrypted cold storage,\n"
        "   consistent with PMLA 2002 AML obligations.\n\n"
        "4. Consent Management\n"
        "   Users provide explicit opt-in consent. Consent is timestamped and version-tagged.\n"
        "   DSAR requests fulfilled within 30 days per GDPR Article 17.\n"
    ),
    "security_incident_procedure.txt": (
        "INFORMATION SECURITY INCIDENT RESPONSE PROCEDURE - v2.1\n\n"
        "1. Scope\n"
        "   Applies to all data breaches involving PHI or PII.\n\n"
        "2. Detection and Triage (0-4 hours)\n"
        "   SOC monitors SIEM alerts 24x7. Suspected breach escalated to CISO and DPO.\n\n"
        "3. Notification (within 72 hours)\n"
        "   Per GDPR Article 33, breaches reported to supervisory authority within 72 hours.\n"
        "   PHI breaches trigger HIPAA Breach Notification Rule obligations.\n\n"
        "4. Encryption Standards\n"
        "   All ePHI protected using AES-256 at rest and TLS 1.3 in transit.\n"
        "   Access governed by RBAC with quarterly access reviews.\n\n"
        "5. Business Associate Agreements\n"
        "   Every vendor with PHI access must sign a BAA before receiving data.\n"
        "   BAA register reviewed bi-annually.\n"
    ),
    "non_compliant_memo.txt": (
        "INTERNAL MEMO - DRAFT (DO NOT DISTRIBUTE)\n\n"
        "Re: Cost Reduction Initiative Q3\n\n"
        "Team proposes migrating customer payment data to Singapore data centre to cut costs by 40%.\n"
        "Migration to be completed without notifying the regulatory team.\n\n"
        "We plan to share patient health records with our analytics partner in Germany\n"
        "without obtaining updated patient consent. Existing generic consent is sufficient.\n\n"
        "KYC verification for premium customers will be fast-tracked with minimal documentation.\n"
        "The compliance team need not be involved.\n\n"
        "Trading window restrictions relaxed for Q4 results to give management flexibility.\n"
        "Insider trading pre-clearance to be waived for the CEO and CFO.\n"
    ),
}

for fname, content in DOCUMENTS.items():
    Path(f"data/documents/{fname}").write_text(content)

df_kyc      = make_kyc_records(300)
df_patients = make_patient_records(200)
df_kyc.to_csv("data/datasets/kyc_records.csv",      index=False)
df_patients.to_csv("data/datasets/patient_records.csv", index=False)

print(f"KYC records     : {len(df_kyc)} rows  | flagged: {df_kyc['flag'].sum()}")
print(f"Patient records : {len(df_patients)} rows  | flagged: {df_patients['flag'].sum()}")
print(f"Policy docs     : {len(DOCUMENTS)} text files -> data/documents/")
df_kyc.head(3)


## Agent 1 — Document Loader & Chunker

In [ ]:
class DocumentLoaderAgent:
    """Loads text files and CSVs, splits them into overlapping chunks."""

    CHUNK_SIZE    = 500
    CHUNK_OVERLAP = 80

    def load_text(self, path: str) -> dict:
        text = Path(path).read_text()
        return {"source": Path(path).name, "content": text,
                "doc_type": "text", "char_count": len(text)}

    def load_csv(self, path: str, sample: int = 15) -> dict:
        df   = pd.read_csv(path)
        text = df.head(sample).to_string(index=False)   # ← was .to_markdown()
        return {"source": Path(path).name, "content": text,
                "doc_type": "csv", "rows": len(df),
                "flags": int(df["flag"].sum()) if "flag" in df.columns else 0}

    def chunk(self, doc: dict) -> list[dict]:
        text, chunks, start, idx = doc["content"], [], 0, 0
        while start < len(text):
            end = min(start + self.CHUNK_SIZE, len(text))
            chunks.append({
                "chunk_id":  f"{doc['source']}_c{idx:03d}",
                "source":    doc["source"],
                "text":      text[start:end].strip(),
                "chunk_idx": idx,
            })
            start += self.CHUNK_SIZE - self.CHUNK_OVERLAP
            idx   += 1
        return chunks

    def run(self, doc_dir="data/documents", csv_dir="data/datasets"):
        docs, chunks = [], []
        for fp in sorted(Path(doc_dir).glob("*.txt")):
            doc = self.load_text(str(fp))
            docs.append(doc); chunks.extend(self.chunk(doc))
        for fp in sorted(Path(csv_dir).glob("*.csv")):
            doc = self.load_csv(str(fp))
            docs.append(doc); chunks.extend(self.chunk(doc))
        print(f"Loaded {len(docs)} documents -> {len(chunks)} chunks")
        return docs, chunks


loader_agent = DocumentLoaderAgent()
documents, all_chunks = loader_agent.run()
print(f"\nSample chunk [{all_chunks[0]['chunk_id']}]:")
print(all_chunks[0]["text"][:200], "...")


## Agent 2 — Rule Matcher

In [ ]:
class RuleMatcherAgent:
    """Fast keyword scan — no LLM needed. Scores each chunk against every rule."""

    def match(self, chunk: dict, threshold: float = 0.10) -> list[dict]:
        results = []
        tl = chunk["text"].lower()
        for rule_id, rule in REGULATORY_RULES.items():
            hits  = [kw for kw in rule["keywords"] if kw.lower() in tl]
            score = len(hits) / len(rule["keywords"])
            if score >= threshold:
                results.append({
                    **chunk,
                    "rule_id":          rule_id,
                    "regulation":       rule["regulation"],
                    "severity":         rule["severity"],
                    "domain":           rule["domain"],
                    "rule_description": rule["description"],
                    "remediation_hint": rule["remediation"],
                    "keyword_hits":     hits,
                    "keyword_score":    round(score, 3),
                })
        return results

    def run(self, chunks: list, threshold: float = 0.10) -> list:
        matches = []
        for chunk in chunks:
            matches.extend(self.match(chunk, threshold))
        print(f"Rule matcher: {len(matches)} chunk-rule matches "
              f"(threshold >= {threshold})")
        return matches


matcher_agent = RuleMatcherAgent()
matches       = matcher_agent.run(all_chunks, threshold=0.10)

df_matches = pd.DataFrame(matches)
print("\nMatches by domain:")
print(df_matches.groupby("domain")["rule_id"].count().to_string())


## Agent 3 — Compliance Checker
**Pydantic AI agent** running on **Qwen3-30B-A3B via vLLM**.  
Returns a validated `ComplianceFinding` object per match.


In [ ]:
# ── Build the Pydantic AI compliance checker agent ────────────────────────────
from pydantic_ai.settings import ModelSettings

compliance_checker_agent = Agent(
    model=agent_model,
    output_type=ComplianceFinding,
    model_settings=ModelSettings(
        max_tokens=1024,        # ← was default (often 256), increase to give room
        temperature=0.1,        # ← low temp = more deterministic JSON output
    ),
    system_prompt=(
        "You are a senior compliance officer at a Big-4 consultancy. "
        "Analyse the provided text excerpt against the given regulatory rule. "
        "Return a structured JSON finding with: "
        "status (VIOLATION/RISK/COMPLIANT/NEEDS_REVIEW), "
        "confidence (0.0-1.0), explanation (1-2 sentences), "
        "remediation (concrete action steps), "
        "evidence (exact quote from text that triggered the finding, or empty string). "
        "Be precise and cite specific regulatory clauses."
    ),
)

def run_compliance_check_sync(match: dict) -> dict:
    """
    Call Qwen3-30B-A3B to analyse one chunk-rule match.
    Falls back to heuristic if vLLM is offline.
    """
    rule = REGULATORY_RULES[match["rule_id"]]

    # ── Heuristic fallback (no vLLM) ─────────────────────────────────────────
    if not VLLM_ONLINE:
        tl        = match["text"].lower()
        red_words = ["without notifying","without consent","need not be involved",
                     "fast-tracked","minimal documentation","relaxed","waived"]
        is_bad    = any(w in tl for w in red_words)
        status    = "VIOLATION" if is_bad else ("RISK" if match["keyword_score"] > 0.3 else "NEEDS_REVIEW")
        conf      = 0.85 if is_bad else 0.55
        return {**match, "status": status, "confidence": conf,
                "explanation": f"Heuristic: {'red-flag language detected' if is_bad else 'keyword match'}.",
                "remediation": rule["remediation"], "evidence": "",
                "risk_score":  round(SEVERITY_SCORE[match["severity"]] * conf, 2)}

    # ── Live Qwen3 call ───────────────────────────────────────────────────────
    prompt = (
        f"RULE ID  : {match['rule_id']}\n"
        f"RULE     : {rule['description']}\n"
        f"SEVERITY : {rule['severity']}\n"
        f"REGULATION: {rule['regulation']}\n\n"
        f"TEXT EXCERPT:\n{match['text'][:400]}"
    )
    try:
        result  = asyncio.get_event_loop().run_until_complete(compliance_checker_agent.run(prompt))
        finding = result.output
        return {
            **match,
            "status":      finding.status,
            "confidence":  finding.confidence,
            "explanation": finding.explanation,
            "remediation": finding.remediation,
            "evidence":    finding.evidence,
            "risk_score":  round(SEVERITY_SCORE[match["severity"]] * finding.confidence, 2),
        }
    except Exception as e:
        print(f"  [WARN] LLM call failed for {match['rule_id']}: {e}")
        return {**match, "status": "NEEDS_REVIEW", "confidence": 0.4,
                "explanation": f"LLM error: {str(e)[:80]}",
                "remediation": rule["remediation"], "evidence": "",
                "risk_score":  SEVERITY_SCORE[match["severity"]] * 0.4}


# ── Run checker on all matches (cap LLM calls to avoid long waits in demo) ────
MAX_LLM_CALLS = 20   # increase for full audit

print(f"Running compliance checker on {len(matches)} matches "
      f"(LLM cap: {MAX_LLM_CALLS})...")

doc_findings = []
llm_used     = 0
for m in matches:
    use_llm  = VLLM_ONLINE and llm_used < MAX_LLM_CALLS
    if not VLLM_ONLINE:
        f = run_compliance_check_sync(m)
    else:
        f = run_compliance_check_sync(m)
        llm_used += 1
    doc_findings.append(f)

df_doc = pd.DataFrame(doc_findings)
print(f"\nCompliance check complete | {len(doc_findings)} findings | LLM calls: {llm_used}")
print("\nStatus distribution:")
print(df_doc["status"].value_counts().to_string())


## Agent 4 — Structured Data Validator

In [ ]:
class DataValidatorAgent:
    """
    Rule-based row-level validation on tabular datasets.
    No LLM needed -- deterministic checks against each rule.
    To use Kaggle datasets: replace CSV paths with the downloaded files.
    """

    CHECKS = {
        "kyc_records.csv": [
            ("RBI_KYC_001",
             lambda r: not r["kyc_verified"],
             "KYC not verified -- identity documents missing"),
            ("RBI_AML_002",
             lambda r: not r["transaction_records_maintained"],
             "Transaction records not maintained for >= 5 years"),
            ("RBI_DATA_003",
             lambda r: not r["data_stored_in_india"],
             "Payment data stored outside India -- violates data localisation"),
        ],
        "patient_records.csv": [
            ("HIPAA_PHI_001",
             lambda r: not r["consent_obtained"],
             "Patient consent not obtained before PHI processing"),
            ("HIPAA_SEC_002",
             lambda r: not r["phi_encrypted"],
             "ePHI not encrypted at rest or in transit"),
            ("HIPAA_BAA_003",
             lambda r: not r["baa_signed"],
             "Business Associate Agreement not signed with vendor"),
            ("GDPR_BREACH_002",
             lambda r: not r["breach_reported_within_72h"],
             "Data breach not reported to DPA within 72 hours"),
        ],
    }

    def run(self, csv_dir: str = "data/datasets") -> list:
        findings = []
        for fname, checks in self.CHECKS.items():
            fp = Path(csv_dir) / fname
            if not fp.exists():
                print(f"  [SKIP] {fname} not found")
                continue
            df = pd.read_csv(fp)
            for rule_id, condition, msg in checks:
                rule      = REGULATORY_RULES[rule_id]
                violators = df[df.apply(condition, axis=1)]
                for _, row in violators.iterrows():
                    findings.append({
                        "source":      fname,
                        "chunk_id":    f"{fname}_{row.name}",
                        "rule_id":     rule_id,
                        "regulation":  rule["regulation"],
                        "severity":    rule["severity"],
                        "domain":      rule["domain"],
                        "status":      "VIOLATION",
                        "confidence":  1.0,
                        "explanation": msg,
                        "remediation": rule["remediation"],
                        "evidence":    "",
                        "risk_score":  float(SEVERITY_SCORE[rule["severity"]]),
                        "record_id":   str(row.get("customer_id", row.get("patient_id",""))),
                    })
        print(f"Data validator: {len(findings)} record-level violations")
        return findings


data_validator_agent = DataValidatorAgent()
data_findings        = data_validator_agent.run()
df_data = pd.DataFrame(data_findings)
print("\nTop violated rules (record level):")
print(df_data["rule_id"].value_counts().head(8).to_string())


## Agent 5 — Audit Report Generator
Uses **Qwen3-30B-A3B** to write the executive summary and remediation plans.


In [ ]:
# ── Pydantic AI agents for report generation ─────────────────────────────────
summary_agent = Agent(
    model=agent_model,
    output_type=AuditSummary,
    model_settings=ModelSettings(
        max_tokens=2048,        # AuditSummary has exec_summary (3 paragraphs)
        temperature=0.1,
    ),
    system_prompt=(
        "You are a Chief Compliance Officer preparing a board-level audit report. "
        "Given compliance findings, produce a structured executive summary with: "
        "overall_risk level, total_findings, critical_count, high_count, "
        "exec_summary (3 paragraphs: risk posture / top issues / next steps), "
        "top_priorities (list of 3 immediate action items). "
        "Be concise and specific. Reference rule IDs."
    ),
)

remediation_agent = Agent(
    model=agent_model,
    output_type=RemediationPlan,
    model_settings=ModelSettings(
        max_tokens=1024,
        temperature=0.1,
    ),
    system_prompt=(
        "You are a compliance remediation specialist. "
        "For the given rule violation, produce a remediation plan with: "
        "priority (IMMEDIATE/SHORT_TERM/LONG_TERM), owner (team/role), "
        "deadline_days (realistic), steps (3-5 concrete action steps). "
        "Be practical and specific to the regulation cited."
    ),
)

SEVERITY_SCORE_MAP = {"critical": 4, "high": 3, "medium": 2, "low": 1}

def compute_risk_metrics(all_df: pd.DataFrame) -> dict:
    sev_scores = all_df["severity"].map(SEVERITY_SCORE_MAP).fillna(1)
    return {
        "total":    int(len(all_df)),
        "critical": int((all_df["severity"] == "critical").sum()),
        "high":     int((all_df["severity"] == "high").sum()),
        "medium":   int((all_df["severity"] == "medium").sum()),
        "low":      int((all_df["severity"] == "low").sum()),
        "avg_score":round(float(sev_scores.mean()), 2) if len(all_df) else 0,
    }

def generate_summary_sync(all_df: pd.DataFrame) -> AuditSummary:
    metrics = compute_risk_metrics(all_df)
    top10   = (all_df.sort_values("risk_score", ascending=False)
                     .head(10)[["rule_id","severity","status","explanation"]]
                     .to_dict("records"))
    findings_text = (
        f"Total: {metrics['total']} | Critical: {metrics['critical']} | "
        f"High: {metrics['high']} | Medium: {metrics['medium']}\n\n"
        "Top findings:\n" +
        "\n".join(f"- [{r['rule_id']}] {r['severity'].upper()}: {r['explanation'][:100]}"
                   for r in top10[:6])
    )
    if not VLLM_ONLINE:
        risk_level = (
            "CRITICAL" if metrics["critical"] >= 5 else
            "HIGH"     if metrics["critical"] >= 1 or metrics["high"] >= 5 else
            "MEDIUM"   if metrics["high"] >= 1 else "LOW"
        )
        return AuditSummary(
            overall_risk   = risk_level,
            total_findings = metrics["total"],
            critical_count = metrics["critical"],
            high_count     = metrics["high"],
            exec_summary   = (
                f"Audit identified {metrics['total']} compliance findings "
                f"({metrics['critical']} critical, {metrics['high']} high). "
                f"Overall risk posture: {risk_level}.\n\n"
                "Critical issues require immediate board attention: data localisation violations, "
                "missing KYC documentation, and PHI encryption gaps present significant regulatory risk.\n\n"
                "Recommended next steps: convene emergency compliance committee, engage external counsel, "
                "and initiate a 30-day remediation sprint."
            ),
            top_priorities = [
                f"Fix {metrics['critical']} critical violations within 72 hours",
                "Engage DPO and CISO for breach assessment",
                "Schedule board-level compliance review",
            ]
        )

    try:
        result = asyncio.get_event_loop().run_until_complete(summary_agent.run(findings_text))
        return result.output
    except Exception as e:
        print(f"  [WARN] Summary agent error: {e}")
        return AuditSummary(
            overall_risk="HIGH", total_findings=metrics["total"],
            critical_count=metrics["critical"], high_count=metrics["high"],
            exec_summary="Summary generation failed. Review findings CSV for details.",
            top_priorities=["Review critical findings","Engage compliance team","Schedule audit"]
        )

def generate_remediation_plans(all_df: pd.DataFrame, top_n: int = 5) -> list[RemediationPlan]:
    top_rules = (all_df[all_df["status"].isin(["VIOLATION","RISK"])]
                     .groupby("rule_id")["risk_score"].sum()
                     .nlargest(top_n).index.tolist())
    plans = []
    for rule_id in top_rules:
        rule    = REGULATORY_RULES.get(rule_id, {})
        row_cnt = int((all_df["rule_id"] == rule_id).sum())
        prompt  = (
            f"RULE: {rule_id} - {rule.get('description','')}\n"
            f"REGULATION: {rule.get('regulation','')}\n"
            f"SEVERITY: {rule.get('severity','')}\n"
            f"VIOLATIONS FOUND: {row_cnt}\n"
            f"INITIAL REMEDIATION HINT: {rule.get('remediation','')}"
        )
        if not VLLM_ONLINE:
            sev = rule.get("severity","high")
            plans.append(RemediationPlan(
                rule_id=rule_id,
                priority="IMMEDIATE" if sev == "critical" else "SHORT_TERM",
                owner={"BFSI":"Compliance Team","Healthcare":"DPO / HIPAA Officer"}.get(rule.get("domain",""),"Compliance Team"),
                deadline_days=7 if sev == "critical" else 30,
                steps=[
                    f"Triage all {row_cnt} affected records",
                    rule.get("remediation","Implement control"),
                    "Document corrective actions in audit log",
                    "Schedule follow-up review in 30 days",
                ]
            ))
        else:
            try:
                result = asyncio.get_event_loop().run_until_complete(remediation_agent.run(prompt))
                plans.append(result.output)
            except Exception as e:
                print(f"  [WARN] Remediation agent error for {rule_id}: {e}")
    return plans


# ── Run both report agents ────────────────────────────────────────────────────
combined_df = pd.concat([df_doc, df_data], ignore_index=True)
print(f"Combined findings: {len(combined_df)} total\n")

print("Generating executive summary...")
audit_summary = generate_summary_sync(combined_df)
print(f"Overall Risk: {audit_summary.overall_risk}")
print(f"Total Findings: {audit_summary.total_findings}  |  Critical: {audit_summary.critical_count}  |  High: {audit_summary.high_count}")
print("\nExecutive Summary:\n")
print(audit_summary.exec_summary)
print("\nTop Priorities:")
for i, p in enumerate(audit_summary.top_priorities, 1):
    print(f"  {i}. {p}")


## Step 13 — Generate Remediation Plans

In [ ]:
print("Generating remediation plans for top violations...\n")
remediation_plans = generate_remediation_plans(combined_df, top_n=5)

for plan in remediation_plans:
    print(f"Rule      : {plan.rule_id}")
    print(f"Priority  : {plan.priority}  |  Owner: {plan.owner}  |  Deadline: {plan.deadline_days} days")
    print("Steps:")
    for i, step in enumerate(plan.steps, 1):
        print(f"  {i}. {step}")
    print()


## Step 14 — Save Audit Reports (JSON + Markdown + CSV)

In [ ]:
report_id = hashlib.md5(datetime.now().isoformat().encode()).hexdigest()[:8].upper()

# ── Domain breakdown ──────────────────────────────────────────────────────────
domain_summary = (combined_df.groupby("domain")
                              .agg(findings=("rule_id","count"),
                                   avg_risk=("risk_score","mean"))
                              .round(2).to_dict("index"))

# ── Full report dict ──────────────────────────────────────────────────────────
report = {
    "report_id":       report_id,
    "generated_at":    datetime.now().isoformat(),
    "audit_period":    "FY 2024-25 Q2",
    "model_used":      "Qwen3-30B-A3B via vLLM (AMD ROCm)",
    "organisation":    "Acme Financial Services & Healthcare Pvt. Ltd.",
    "vllm_online":     VLLM_ONLINE,
    "risk_summary":    {
        "overall_risk":  audit_summary.overall_risk,
        "total":         audit_summary.total_findings,
        "critical":      audit_summary.critical_count,
        "high":          audit_summary.high_count,
        "exec_summary":  audit_summary.exec_summary,
        "top_priorities":audit_summary.top_priorities,
    },
    "domain_summary":  domain_summary,
    "top_findings":    (combined_df.sort_values("risk_score", ascending=False)
                                   .head(10)[["rule_id","source","severity","status",
                                              "explanation","risk_score"]]
                                   .to_dict("records")),
    "remediation_plans": [p.model_dump() for p in remediation_plans],
    "records_scanned": {
        "documents":       len(documents),
        "kyc_records":     len(df_kyc),
        "patient_records": len(df_patients),
    },
}

# ── JSON ──────────────────────────────────────────────────────────────────────
json_path = f"reports/audit_{report_id}.json"
with open(json_path, "w") as f:
    json.dump(report, f, indent=2, default=str)

# ── Markdown ──────────────────────────────────────────────────────────────────
def make_md_report(report: dict, df: pd.DataFrame) -> str:
    rs   = report["risk_summary"]
    top  = report["top_findings"]
    lines = [
        f"# Compliance Audit Report - {report['report_id']}",
        f"**Organisation:** {report['organisation']}",
        f"**Period:** {report['audit_period']}",
        f"**Model:** {report['model_used']}",
        f"**Generated:** {report['generated_at']}",
        "", "---", "## Executive Summary", "",
        rs["exec_summary"], "", "---", "## Risk Dashboard", "",
        "| Metric | Value |", "|--------|-------|",
        f"| Overall Risk Level | **{rs['overall_risk']}** |",
        f"| Total Findings | {rs['total']} |",
        f"| Critical | {rs['critical']} |",
        f"| High | {rs['high']} |",
        "", "---", "## Top Priorities", "",
    ]
    for i, p in enumerate(rs["top_priorities"], 1):
        lines.append(f"{i}. {p}")
    lines += ["", "---", "## Domain Breakdown", "",
              "| Domain | Findings | Avg Risk |", "|--------|----------|----------|"]
    for domain, vals in report["domain_summary"].items():
        lines.append(f"| {domain} | {vals['findings']} | {vals['avg_risk']:.2f} |")
    lines += ["", "---", "## Top 10 Findings", "",
              "| # | Rule | Source | Severity | Status | Explanation |",
              "|---|------|--------|----------|--------|-------------|"]
    for i, f in enumerate(top, 1):
        lines.append(f"| {i} | {f['rule_id']} | {f['source']} | "
                     f"{f['severity'].upper()} | {f['status']} | {str(f['explanation'])[:80]}... |")
    lines += ["", "---", "## Remediation Checklist", ""]
    for plan in report["remediation_plans"]:
        lines.append(f"- [ ] **{plan['rule_id']}** ({plan['priority']}, {plan['deadline_days']}d) -- {plan['steps'][0]}")
    lines.append("\n---\n*Generated by Multi-Agent AI Compliance Validator | Qwen3-30B-A3B on AMD ROCm*")
    return "\n".join(lines)

md_path  = f"reports/audit_{report_id}.md"
csv_path = f"reports/findings_{report_id}.csv"
Path(md_path).write_text(make_md_report(report, combined_df))
keep_cols = [c for c in ["rule_id","source","domain","severity","status",
                          "confidence","risk_score","explanation","remediation"]
             if c in combined_df.columns]
combined_df[keep_cols].to_csv(csv_path, index=False)

print(f"Reports saved:")
print(f"  JSON  -> {json_path}")
print(f"  MD    -> {md_path}")
print(f"  CSV   -> {csv_path}")


## Step 15 — Compliance Dashboard

In [ ]:
import matplotlib
matplotlib.use("Agg")   # use "inline" in standard Jupyter
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.patch.set_facecolor("#F8F9FA")
fig.suptitle(f"Compliance Audit Dashboard -- {report_id}", fontsize=14, fontweight="bold", y=1.01)

COLORS = {"critical":"#E24B4A","high":"#EF9F27","medium":"#378ADD","low":"#639922"}
DOMAIN_COLORS = ["#534AB7","#1D9E75","#D85A30","#BA7517"]

# ── Chart 1: Findings by severity ─────────────────────────────────────────────
sev = combined_df["severity"].value_counts().reindex(["critical","high","medium","low"]).fillna(0)
sev.plot.bar(ax=axes[0,0], color=[COLORS[s] for s in sev.index],
             edgecolor="white", width=0.6)
axes[0,0].set_title("Findings by Severity", fontweight="bold")
axes[0,0].set_xlabel(""); axes[0,0].tick_params(axis="x", rotation=0)
for bar, val in zip(axes[0,0].patches, sev.values):
    axes[0,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                   f"{int(val)}", ha="center", fontsize=10, fontweight="bold")

# ── Chart 2: Domain distribution ──────────────────────────────────────────────
dom = combined_df["domain"].value_counts()
axes[0,1].pie(dom, labels=dom.index, colors=DOMAIN_COLORS[:len(dom)],
              autopct="%1.0f%%", startangle=140,
              wedgeprops={"edgecolor":"white","linewidth":1.5})
axes[0,1].set_title("Findings by Domain", fontweight="bold")

# ── Chart 3: Risk score by rule ────────────────────────────────────────────────
top_rules = combined_df.groupby("rule_id")["risk_score"].sum().nlargest(8).sort_values()
colors_bar = [COLORS.get(
    REGULATORY_RULES.get(r,{}).get("severity","low"), "#888") for r in top_rules.index]
top_rules.plot.barh(ax=axes[1,0], color=colors_bar, edgecolor="white", width=0.6)
axes[1,0].set_title("Cumulative Risk Score - Top 8 Rules", fontweight="bold")
axes[1,0].set_xlabel("Cumulative Risk Score")

# ── Chart 4: Status breakdown ──────────────────────────────────────────────────
status_colors = {"VIOLATION":"#E24B4A","RISK":"#EF9F27",
                 "NEEDS_REVIEW":"#378ADD","COMPLIANT":"#639922"}
stat = combined_df["status"].value_counts()
axes[1,1].bar(stat.index, stat.values,
              color=[status_colors.get(s,"#888") for s in stat.index],
              edgecolor="white", width=0.6)
axes[1,1].set_title("Finding Status Distribution", fontweight="bold")
axes[1,1].tick_params(axis="x", rotation=15)
for bar, val in zip(axes[1,1].patches, stat.values):
    axes[1,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                   f"{int(val)}", ha="center", fontsize=10, fontweight="bold")

# Legend
patches = [mpatches.Patch(color=v, label=k) for k,v in COLORS.items()]
fig.legend(handles=patches, loc="upper right", title="Severity", framealpha=0.8)

plt.tight_layout(pad=2.5)
chart_path = f"reports/dashboard_{report_id}.png"
plt.savefig(chart_path, dpi=130, bbox_inches="tight")
plt.show()
print(f"Dashboard saved -> {chart_path}")


## Step 16 — Interactive Compliance Q&A
Ask natural-language questions about the audit findings — answered by **Qwen3-30B-A3B**.


In [ ]:
qa_agent = Agent(
    model=agent_model,
    output_type=str,
    system_prompt=(
        "You are an expert compliance advisor. Answer questions about audit findings "
        "using the provided context. Be concise, cite rule IDs, and give actionable advice. "
        "If the context does not contain enough information, say so clearly."
    ),
)

def ask_compliance_question(question: str, top_k: int = 6) -> str:
    """Answer a natural-language compliance question using audit findings as context."""
    # Retrieve relevant findings via keyword search
    kw      = question.lower().split()
    mask    = combined_df.apply(
                  lambda row: any(k in str(row.values).lower() for k in kw), axis=1)
    context = (combined_df[mask]
                   .head(top_k)[["rule_id","explanation","remediation","severity","status"]]
                   .to_string(index=False))

    prompt = (
        f"QUESTION: {question}\n\n"
        f"AUDIT CONTEXT (relevant findings):\n{context}\n\n"
        "Provide a concise, actionable answer."
    )

    if not VLLM_ONLINE:
        return (
            f"[DRY-RUN -- vLLM offline]\n\n"
            f"Relevant findings for: '{question}'\n\n{context}"
        )
    try:
        result = asyncio.get_event_loop().run_until_complete( qa_agent.run(prompt))
        return result.output
    except Exception as e:
        return f"Q&A agent error: {e}\n\nRaw context:\n{context}"


# ── Demo questions ────────────────────────────────────────────────────────────
DEMO_QUESTIONS = [
    "What are the most critical HIPAA violations found and what should we do immediately?",
    "Which RBI rules are we non-compliant with and what is the financial risk?",
    "What GDPR actions must be taken within 72 hours?",
    "Summarise the top 3 remediation priorities for the CISO.",
]

for q in DEMO_QUESTIONS:
    print(f"Q: {q}")
    print("-" * 65)
    print(ask_compliance_question(q))
    print()


## Step 17 — Full Pipeline Orchestrator

One function to run the entire 5-agent pipeline end-to-end.


In [ ]:
def run_compliance_pipeline(
    doc_dir:           str   = "data/documents",
    csv_dir:           str   = "data/datasets",
    keyword_threshold: float = 0.10,
    max_llm_calls:     int   = 20,
) -> dict:
    """
    End-to-end multi-agent compliance validation pipeline.

    Agents
    ------
    1. DocumentLoaderAgent    -- ingest & chunk documents / CSVs
    2. RuleMatcherAgent       -- keyword-based rule matching
    3. ComplianceCheckerAgent -- Qwen3-30B-A3B verdict per finding
    4. DataValidatorAgent     -- row-level tabular validation
    5. ReportGeneratorAgent   -- LLM executive summary + remediation plans

    Returns
    -------
    dict: report, combined_df, json_path, md_path, csv_path
    """
    t0 = datetime.now()
    print("=" * 60)
    print("  MULTI-AGENT COMPLIANCE VALIDATOR")
    print("  Model : Qwen3-30B-A3B  |  Backend: vLLM (AMD ROCm)")
    print("=" * 60)

    print("\n[1/5] Document Loader...")
    _loader = DocumentLoaderAgent()
    _docs, _chunks = _loader.run(doc_dir, csv_dir)

    print("[2/5] Rule Matcher...")
    _matcher = RuleMatcherAgent()
    _matches = _matcher.run(_chunks, keyword_threshold)

    print("[3/5] Compliance Checker (Qwen3)...")
    _doc_finds, _llm = [], 0
    for m in _matches:
        _doc_finds.append(run_compliance_check_sync(m))
        if VLLM_ONLINE: _llm += 1
        if _llm >= max_llm_calls: break

    print("[4/5] Data Validator...")
    _dv = DataValidatorAgent()
    _data_finds = _dv.run(csv_dir)

    print("[5/5] Report Generator (Qwen3)...")
    _all_df  = pd.concat([pd.DataFrame(_doc_finds),
                           pd.DataFrame(_data_finds)], ignore_index=True)
    _summary = generate_summary_sync(_all_df)
    _plans   = generate_remediation_plans(_all_df, top_n=5)

    _rid     = hashlib.md5(datetime.now().isoformat().encode()).hexdigest()[:8].upper()
    _report  = {
        "report_id": _rid, "generated_at": datetime.now().isoformat(),
        "model_used": "Qwen3-30B-A3B via vLLM (AMD ROCm)",
        "risk_summary": {
            "overall_risk": _summary.overall_risk,
            "total": _summary.total_findings,
            "critical": _summary.critical_count,
            "high": _summary.high_count,
            "exec_summary": _summary.exec_summary,
        },
        "domain_summary": (_all_df.groupby("domain")
                                   .agg(findings=("rule_id","count"),
                                        avg_risk=("risk_score","mean"))
                                   .round(2).to_dict("index")),
        "remediation_plans": [p.model_dump() for p in _plans],
    }
    _jp = f"reports/audit_{_rid}.json"
    _mp = f"reports/audit_{_rid}.md"
    _cp = f"reports/findings_{_rid}.csv"
    with open(_jp, "w") as f: json.dump(_report, f, indent=2, default=str)
    Path(_mp).write_text(make_md_report(_report, _all_df))
    _all_df.to_csv(_cp, index=False)

    elapsed = (datetime.now() - t0).total_seconds()
    print(f"\n{'='*60}")
    print(f"  DONE in {elapsed:.1f}s | Risk: {_summary.overall_risk} | "
          f"Findings: {_summary.total_findings}")
    print(f"  JSON  -> {_jp}")
    print(f"  MD    -> {_mp}")
    print("=" * 60)
    return {"report": _report, "combined_df": _all_df,
            "json_path": _jp, "md_path": _mp, "csv_path": _cp}


# ── Uncomment to run fresh end-to-end ─────────────────────────────────────────
# result = run_compliance_pipeline(max_llm_calls=20)
print("Orchestrator ready. Call run_compliance_pipeline() to execute.")


## Next Steps & Extensions

### Swap in real datasets
| Dataset | Source | Replace |
|---------|--------|---------|
| IBM AML 6M transactions | [Kaggle](https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml) | `kyc_records.csv` |
| MIMIC-III clinical notes | [PhysioNet](https://physionet.org/content/mimiciii/) | `patient_records.csv` |
| SEC financial fraud | [Kaggle](https://www.kaggle.com/datasets/meirion/financial-statement-fraud-detection-dataset) | add new CSV check |
| Synthea synthetic patients | [GitHub](https://github.com/synthetichealth/synthea) | `patient_records.csv` |

### vLLM performance tuning on AMD
```bash
# Multi-GPU (tensor parallel across 2x MI300X)
vllm serve Qwen/Qwen3-30B-A3B \
    --tensor-parallel-size 2 \
    --gpu-memory-utilization 0.92 \
    --max-num-seqs 32 \
    --enable-auto-tool-choice \
    --tool-call-parser hermes \
    --trust-remote-code
```

### Production enhancements
- **Async agents** — use `compliance_checker_agent.run()` (async) + `asyncio.gather()` for 10x throughput
- **Vector search** — add ChromaDB + sentence-transformers for semantic chunk retrieval
- **Streaming** — use `agent.run_stream()` for real-time audit progress in a dashboard
- **Structured tool calls** — expose rule bank as a tool so the model can query it dynamically
- **Scheduler** — wrap `run_compliance_pipeline()` in Airflow or Prefect for daily audits
- **API server** — expose via FastAPI with `/audit`, `/ask`, `/report` endpoints
